In [ ]:
# ── MLOps bootstrap (auto-injected by inject_mlops_cell.py) ──────────────────
import os, warnings, mlflow
warnings.filterwarnings("ignore")

SEED = 42
import random, numpy as np
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED)
except ImportError:
    pass
try:
    import tensorflow as tf; tf.random.set_seed(SEED)
except ImportError:
    pass

_nb_name = os.path.basename(os.path.abspath("__file__") if "__file__" in dir() else ".").replace(".ipynb","")
mlflow.set_tracking_uri("sqlite:///" + str(Path(__file__).parent.parent.parent / "mlflow.db")
                        if "__file__" in dir() else "sqlite:///mlflow.db")
_exp = mlflow.set_experiment(_nb_name or "unnamed_notebook")
print(f"MLflow experiment: {_exp.name}")


# 🔄 RAG with Query Rewriting

## What We're Building

A RAG pipeline that **rewrites ambiguous user questions** before sending
them to the retriever. This dramatically improves retrieval quality.

## The Problem

Users ask bad questions:
- **Vague**: "How does it work?" (what is 'it'?)
- **Conversational**: "What about the pricing?" (pricing of what?)
- **Multi-intent**: "Compare A vs B and also how to set up C"

These queries retrieve irrelevant documents because the retriever takes
them literally.

## Three Rewriting Strategies

| Strategy | How it works | When to use |
|----------|-------------|------------|
| **Clarification** | Make vague queries specific | Ambiguous questions |
| **Step-back** | Ask a broader question first | Too-specific queries |
| **Multi-query** | Generate multiple search queries | Complex questions |

## Stack
- **LangChain** — chain orchestration
- **ChromaDB** — vector retrieval
- **Ollama** — local LLM for rewriting + answering

## Step 1 — Imports & Knowledge Base

In [ ]:
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_community.vectorstores import Chroma
from langchain.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain.schema import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter

print("All imports successful!")

In [ ]:
# Knowledge base about a fictional SaaS product
knowledge_base = """
# CloudSync Platform Documentation

## Overview
CloudSync is a real-time data synchronization platform for enterprises.
It syncs databases, APIs, and file systems across cloud providers.
Founded in 2020, it now serves 5,000+ enterprise customers.

## Pricing
CloudSync offers three pricing tiers:
- Starter: $49/month — up to 5 data sources, 100K sync operations/month
- Professional: $199/month — up to 25 sources, 1M operations, priority support
- Enterprise: Custom pricing — unlimited sources, SLA guarantees, dedicated support
All plans include a 14-day free trial. Annual billing gets 20% discount.

## Authentication
CloudSync uses OAuth 2.0 for API authentication. Users generate API keys
from the dashboard. Keys can be scoped to read-only, write, or admin.
SSO is available via SAML for Enterprise customers. Okta, Azure AD, and
Google Workspace are supported as identity providers.

## Data Sources
Supported databases: PostgreSQL, MySQL, MongoDB, DynamoDB, Snowflake.
Supported file systems: S3, GCS, Azure Blob Storage.
Supported APIs: REST, GraphQL, Webhooks.
Custom connectors can be built using the CloudSync SDK (Python, Node.js, Go).

## Sync Modes
1. Full Sync: Copies all data from source to destination. Best for initial setup.
2. Incremental Sync: Only transfers changed data. Uses change data capture (CDC).
3. Real-time Sync: Continuous streaming with sub-second latency. Requires CDC.

## Error Handling
CloudSync retries failed operations with exponential backoff (1s, 2s, 4s, 8s).
Dead-letter queues capture events that fail after 5 retries.
Alert notifications can be sent via email, Slack, or PagerDuty.

## Performance
Throughput: Up to 100K events/second on Enterprise tier.
Latency: P50 = 50ms, P95 = 200ms, P99 = 500ms for real-time sync.
Data is encrypted in transit (TLS 1.3) and at rest (AES-256).

## Monitoring
The CloudSync dashboard shows real-time metrics: sync lag, throughput,
error rates, and data volume. Historical metrics are retained for 90 days.
Prometheus metrics endpoint is available for custom monitoring.

## Compliance
CloudSync is SOC 2 Type II certified, GDPR compliant, and HIPAA eligible.
Data residency controls allow pinning data to specific AWS/GCP regions.
"""

# Split and index
splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=50, separators=["\n## ", "\n\n", "\n"])
docs = splitter.create_documents([knowledge_base], metadatas=[{"source": "cloudsync_docs.md"}])

embeddings = OllamaEmbeddings(model="nomic-embed-text-v2-moe")
vectorstore = Chroma.from_documents(docs, embeddings, collection_name="query_rewriting")
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

llm = ChatOllama(model="qwen3.5:9b", temperature=0.1)

print(f"Indexed {len(docs)} chunks")

## Step 2 — Strategy 1: Query Clarification

Rewrite the query to be more specific and retrieval-friendly.

In [ ]:
clarify_prompt = ChatPromptTemplate.from_template(
    """You are a query rewriting assistant. Rewrite the user's question to be
more specific, clear, and optimized for document retrieval.

Rules:
- Resolve ambiguous pronouns ("it", "they", "that")
- Add specific technical terms when implied
- Keep the rewritten query concise (1-2 sentences)
- Do NOT answer the question — only rewrite it
- Output ONLY the rewritten query, nothing else

Original question: {question}

Rewritten query:"""
)

clarify_chain = clarify_prompt | llm | StrOutputParser()


# Test it
test_queries = [
    "How does it work?",
    "What about the pricing?",
    "Can I connect my database?",
    "Is it secure?",
]

print("Strategy 1: Query Clarification\n")
for q in test_queries:
    rewritten = clarify_chain.invoke({"question": q})
    print(f"  Original:  {q}")
    print(f"  Rewritten: {rewritten}")
    print()

## Step 3 — Strategy 2: Step-Back Prompting

When a query is too narrow, ask a **broader question** first to get
background context, then answer the specific question.

In [ ]:
stepback_prompt = ChatPromptTemplate.from_template(
    """You are a query rewriting assistant. Given a specific question, generate
a broader "step-back" question that would retrieve useful background context.

The step-back question should:
- Be more general than the original
- Capture the broader topic or category
- Help retrieve foundational context

Output ONLY the step-back question, nothing else.

Specific question: {question}

Step-back question:"""
)

stepback_chain = stepback_prompt | llm | StrOutputParser()


# Test it
specific_queries = [
    "What is the P99 latency for real-time sync?",
    "Does CloudSync support Okta SSO?",
    "What happens after 5 retries fail?",
]

print("Strategy 2: Step-Back Prompting\n")
for q in specific_queries:
    stepback = stepback_chain.invoke({"question": q})
    print(f"  Original:  {q}")
    print(f"  Step-back: {stepback}")
    print()

## Step 4 — Strategy 3: Multi-Query Expansion

Generate **multiple search queries** from one question to improve
recall — each sub-query might retrieve different relevant chunks.

In [ ]:
multiquery_prompt = ChatPromptTemplate.from_template(
    """You are a query expansion assistant. Given a user question, generate
3 different search queries that would help find relevant information.

Each query should approach the question from a different angle:
- Query 1: Direct reformulation
- Query 2: Related concepts or synonyms
- Query 3: Broader context query

Output ONLY the 3 queries, one per line, numbered 1-3.

User question: {question}

Search queries:"""
)

multiquery_chain = multiquery_prompt | llm | StrOutputParser()


def parse_multi_queries(response: str) -> list[str]:
    """Parse numbered queries from LLM response."""
    queries = []
    for line in response.strip().split("\n"):
        line = line.strip()
        if line and line[0].isdigit():
            # Remove leading number and punctuation
            cleaned = line.lstrip("0123456789.-) ").strip()
            if cleaned:
                queries.append(cleaned)
    return queries[:3]


# Test it
complex_query = "How do I migrate from a competitor to CloudSync and what certifications does it have?"
response = multiquery_chain.invoke({"question": complex_query})
sub_queries = parse_multi_queries(response)

print("Strategy 3: Multi-Query Expansion\n")
print(f"Original: {complex_query}\n")
for i, sq in enumerate(sub_queries, 1):
    print(f"  Sub-query {i}: {sq}")

## Step 5 — Complete Pipeline: Rewrite → Retrieve → Answer

Putting it all together with side-by-side comparison.

In [ ]:
answer_prompt = ChatPromptTemplate.from_template(
    """Answer the question using ONLY the provided context.
If the context doesn't contain the answer, say so.

Context:
{context}

Question: {question}

Answer:"""
)
answer_chain = answer_prompt | llm | StrOutputParser()


def rag_without_rewrite(question: str) -> str:
    """Standard RAG — no query rewriting."""
    docs = retriever.invoke(question)
    context = "\n\n".join(d.page_content for d in docs)
    return answer_chain.invoke({"context": context, "question": question})


def rag_with_clarification(question: str) -> str:
    """RAG with query clarification."""
    rewritten = clarify_chain.invoke({"question": question})
    docs = retriever.invoke(rewritten)
    context = "\n\n".join(d.page_content for d in docs)
    return answer_chain.invoke({"context": context, "question": question})


def rag_with_multiquery(question: str) -> str:
    """RAG with multi-query expansion — union of results."""
    response = multiquery_chain.invoke({"question": question})
    sub_queries = parse_multi_queries(response)
    all_docs = []
    seen = set()
    for sq in sub_queries:
        docs = retriever.invoke(sq)
        for d in docs:
            if d.page_content not in seen:
                seen.add(d.page_content)
                all_docs.append(d)
    context = "\n\n".join(d.page_content for d in all_docs[:5])
    return answer_chain.invoke({"context": context, "question": question})


print("All pipelines ready!")

In [ ]:
# Compare on an ambiguous query
test_q = "How does it handle failures?"

print(f"❓ Question: {test_q}")
print("=" * 60)

print("\n📌 Without Rewriting:")
print(rag_without_rewrite(test_q))

print("\n📌 With Clarification:")
print(rag_with_clarification(test_q))

print("\n📌 With Multi-Query:")
print(rag_with_multiquery(test_q))

In [ ]:
# Test on a multi-intent question
test_q2 = "What databases are supported and how much does it cost?"

print(f"❓ Question: {test_q2}")
print("=" * 60)

print("\n📌 Without Rewriting:")
print(rag_without_rewrite(test_q2))

print("\n📌 With Multi-Query:")
print(rag_with_multiquery(test_q2))

## 🧠 Key Concepts Recap

| Strategy | When to use | Tradeoff |
|----------|------------|----------|
| **Clarification** | Vague/ambiguous queries | 1 extra LLM call |
| **Step-back** | Too-specific queries | May retrieve too-broad context |
| **Multi-query** | Complex multi-intent queries | 3x retrieval cost |
| **No rewrite** | Clear, specific queries | Works fine as-is |

## 🔧 Customization Ideas

- **Auto-detect strategy**: Use LLM to classify query type and pick strategy
- **HyDE (Hypothetical Doc Embeddings)**: Generate a fake answer, embed that
- **Query routing**: Different retrievers for different question types
- **DSPy optimization**: Use DSPy to auto-optimize rewriting prompts